# Boosting: trees that fix each other's mistakes
**Machine Learning · Week 8 (The rest of the classical toolbox) · Day 2**

Lesson page: https://sara-aminpour.com/tutorials/ml/w8d2.html

**Boosting** builds a team of small trees one after another. Each new tree is trained on what the team
still gets wrong, and joins with a small weight (the learning rate). Here it is by hand, on a wiggly curve.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor

rng = np.random.default_rng(0)
x = np.sort(rng.uniform(0, 6, 80))
y = np.sin(x) + 0.3 * x + rng.normal(0, 0.25, 80)
X = x[:, None]                                   # scikit-learn wants a 2D array: (samples, features)

def boost(X, y, rounds, lr=0.3, depth=1):
    """Gradient boosting for squared error: every tree fits the current residuals."""
    pred = np.full(len(y), y.mean())             # start from the average
    trees = []
    for _ in range(rounds):
        residual = y - pred                      # what the team still gets wrong
        tree = DecisionTreeRegressor(max_depth=depth).fit(X, residual)
        pred += lr * tree.predict(X)             # join with a small weight
        trees.append(tree)
    return y.mean(), trees

grid = np.linspace(0, 6, 400)[:, None]
fig, axes = plt.subplots(1, 4, figsize=(13, 3.2), sharey=True, layout="constrained")
for ax, rounds in zip(axes, [1, 5, 20, 100]):
    start, trees = boost(X, y, rounds)
    curve = start + 0.3 * sum(t.predict(grid) for t in trees)
    ax.scatter(x, y, s=12, color="#5577f5")
    ax.plot(grid[:, 0], curve, color="#ff8a50", lw=2.5)
    ax.set_title(f"{rounds} stump{'s' if rounds > 1 else ''}")
plt.show()

## AdaBoost: shout louder about the mistakes
The first boosting algorithm (1995) works with weights instead of residuals: after each round, the
misclassified points get heavier, so the next stump has to care about them. Here it is in ten lines.
Point size = weight after 10 rounds.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.tree import DecisionTreeClassifier

Xm, ym = make_moons(n_samples=200, noise=0.25, random_state=1)
s = np.where(ym == 1, 1, -1)                     # AdaBoost likes labels -1 and +1
w = np.full(len(s), 1 / len(s))                  # every point starts equally important
stumps, alphas = [], []
for _ in range(10):
    stump = DecisionTreeClassifier(max_depth=1).fit(Xm, s, sample_weight=w)
    miss = stump.predict(Xm) != s
    err = w[miss].sum()
    alpha = 0.5 * np.log((1 - err) / err)        # a better stump gets a bigger say
    w *= np.exp(alpha * np.where(miss, 1, -1))   # mistakes get heavier, the rest lighter
    w /= w.sum()
    stumps.append(stump)
    alphas.append(alpha)

score = sum(a * st.predict(Xm) for a, st in zip(alphas, stumps))
print(f"training accuracy after 10 rounds: {np.mean(np.sign(score) == s):.2f}")
plt.figure(figsize=(5.5, 3.6))
plt.scatter(Xm[:, 0], Xm[:, 1], s=4000 * w, c=np.where(ym == 0, "#ff8a50", "#5577f5"), alpha=0.8)
plt.title("After 10 rounds: the hard points weigh the most")
plt.show()

## Gradient boosting in scikit-learn, and how many rounds
A harder problem: 10 features, and class 1 is whatever lies outside a sphere. A single tree struggles.
`staged_predict` replays the model after every round, so you can watch the test accuracy grow and see how
the learning rate trades speed for care.

In [ ]:
from sklearn.datasets import make_hastie_10_2
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier

Xh, yh = make_hastie_10_2(n_samples=4000, random_state=0)
Xh_tr, Xh_te, yh_tr, yh_te = train_test_split(Xh, yh, test_size=0.3, random_state=0)
print(f"one deep tree:       {DecisionTreeClassifier(random_state=0).fit(Xh_tr, yh_tr).score(Xh_te, yh_te):.3f}")
print(f"forest of 300 trees: {RandomForestClassifier(n_estimators=300, random_state=0).fit(Xh_tr, yh_tr).score(Xh_te, yh_te):.3f}")
plt.figure(figsize=(6.4, 3.6))
for lr, color in [(1.0, "#e45fa3"), (0.3, "#ff8a50"), (0.05, "#17a98c")]:
    gb = GradientBoostingClassifier(n_estimators=400, learning_rate=lr, max_depth=2, random_state=0).fit(Xh_tr, yh_tr)
    acc = [np.mean(p == yh_te) for p in gb.staged_predict(Xh_te)]
    plt.plot(range(1, 401), acc, color=color, label=f"learning rate {lr}: best {max(acc):.3f}")
plt.xlabel("number of trees")
plt.ylabel("test accuracy")
plt.ylim(0.5, 1)
plt.legend()
plt.title("Gradient boosting, depth-2 trees")
plt.show()

## The fast version you'd use for real
`HistGradientBoostingClassifier` (like the famous XGBoost and LightGBM libraries) bins every feature first,
which makes it fast on large tables. With `early_stopping=True` it keeps a validation slice aside and stops
when it stops improving.

In [ ]:
import time
from sklearn.ensemble import HistGradientBoostingClassifier

t0 = time.perf_counter()
hgb = HistGradientBoostingClassifier(max_iter=1000, learning_rate=0.1, early_stopping=True, random_state=0).fit(Xh_tr, yh_tr)
print(f"stopped after {hgb.n_iter_} rounds, in {time.perf_counter() - t0:.1f} s")
print(f"test accuracy: {hgb.score(Xh_te, yh_te):.3f}")

## Your turn: deeper trees or more of them?
Train `GradientBoostingClassifier(n_estimators=300, learning_rate=0.1, ...)` on the sphere problem with
`max_depth` 1, 2, 3 and 5. For each, print the best test accuracy over the rounds and the round where it happened.
(Strictly, picking the best round should use a validation set; here we just want to see the pattern.)

In [ ]:
for depth in [1, 2, 3, 5]:
    gb = None     # TODO: fit a GradientBoostingClassifier with this max_depth (random_state=0)
    acc = None    # TODO: the list of test accuracies after each round, from gb.staged_predict(Xh_te)
    print(f"depth {depth}: best accuracy {None} after {None} trees")

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
for depth in [1, 2, 3, 5]:
    gb = GradientBoostingClassifier(n_estimators=300, learning_rate=0.1, max_depth=depth, random_state=0).fit(Xh_tr, yh_tr)
    acc = [np.mean(p == yh_te) for p in gb.staged_predict(Xh_te)]
    print(f"depth {depth}: best accuracy {max(acc):.3f} after {int(np.argmax(acc)) + 1} trees")
print("Stumps learn slowly (still climbing at 300), depth 3 is best, and depth 5 peaks earlier and lower:")
print("deeper trees learn faster per round, but start fitting the noise sooner.")